### 1. Basic Tasks

In [0]:
from pyspark.sql.functions import *

In [0]:
# 1.
df = spark.read.csv('/Volumes/cyntexa_dev/sales/raw/ecommerce.csv', header = True, inferSchema = True)

In [0]:
from pyspark.sql.functions import col

df.filter(
    col("email").isNull() |
    col("quantity").isNull() |
    col("price").isNull() |
    col("discount").isNull()
).show()

In [0]:
df.count()

In [0]:
df.distinct().count()

In [0]:
df = df.dropDuplicates()

In [0]:
# 2.
df = df.withColumnsRenamed({
    "customer_id":"cust_id",
    "customer_name":"cust_name",
    "product_id":"prod_id",
    "product_name":"prod_name"
})

#### 3.
Connected databricks repo to git provider and made firt commit containg cleaning data notebook

### 2. Intermediate Tasks

In [0]:
# 4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove exact duplicates, and sort by order_date.
df_cleaned = df.fillna({"email":"unknown@email.com","discount":0}) \
    .dropDuplicates() \
        .dropna("all") \
            .orderBy("order_date")

In [0]:
# 5. Perform an aggregation (revenue by category or region) and a join against a second small reference table (e.g., customers or regions).
df.groupBy("category").agg(sum("price")).show()

### 6.
Create a freature branch in Databricks Repo and updates cleaning logic. The change was committed to the feature branch and a pull request was opened to review the changes before merging them into the main branch 

### 3. Advanced Tasks

In [0]:
# 7.
df = spark.read.csv("/Volumes/cyntexa_dev/bronze/raw/dirty_orders.csv",
                    header = True,
                    inferSchema = True)

df.printSchema()

In [0]:
df.select("discount_amount", "total_amount").display()

In [0]:
df_cleaned = df.withColumn("discount_amount", regexp_replace(col("discount_amount"), "[$₹€£,]","")) \
    .withColumn("total_amount", regexp_replace(col("total_amount"), "[$₹€£,]",""))

In [0]:
from pyspark.sql.functions import expr

df_cleaned = df_cleaned.withColumn("discount_amount", expr("try_cast(discount_amount as double)")) \
    .withColumn("total_amount", expr("try_cast(total_amount as double)")) \
    .dropna(subset=["discount_amount", "total_amount"])

df_cleaned.printSchema()

In [0]:
df_cleaned.select("discount_amount","total_amount").display()

8. Set up a branching strategy (dev/main) for the Cyntexa analytics repo and write a short guide for teammates on the pull-request review workflow before merging into main.


## Branching Strategy

We use a **two-branch strategy** for the Cyntexa analytics repository:

### Branch Structure

- **`main`** - Production-ready code
  - Always stable and deployable
  - Protected branch (requires PR approval)
  - All merges must come through pull requests
  
- **`dev`** - Development branch
  - Integration branch for ongoing work
  - Contains latest development changes
  - Should be relatively stable but may have work in progress

### Working with Branches

1. **Feature branches**: Create from `dev` for new features or bug fixes
   ```bash
   git checkout dev
   git pull origin dev
   git checkout -b feature/your-feature-name
   ```

2. **Naming conventions**:
   - Features: `feature/description` (e.g., `feature/customer-segmentation`)
   - Bug fixes: `bugfix/description` (e.g., `bugfix/null-handling`)
   - Hotfixes: `hotfix/description
   ` (for urgent production fixes)

---

## Pull Request Review Workflow

### Creating a Pull Request

1. **Push your feature branch**
   ```bash
   git push origin feature/your-feature-name
   ```

2. **Open a PR** on GitHub/GitLab/Azure DevOps:
   - **Base branch**: `dev` (for most PRs)
   - **Compare branch**: your feature branch
   - Write a clear title and description
   - Link related issues/tickets

3. **PR Description should include**:
   - What: Summary of changes
   - Why: Business justification or problem solved
   - How: Technical approach (if complex)
   - Testing: How you verified the changes

### Review Process

1. **Request reviewers**: Tag at least one teammate
2. **Automated checks**: Ensure all CI/CD checks pass
3. **Reviewer responsibilities**:
   - Check code quality and readability
   - Verify logic correctness
   - Test locally if needed
   - Provide constructive feedback
4. **Author responsibilities**:
   - Address all review comments
   - Re-request review after making changes

### Merging to `main`

**Only merge to `main` from `dev`** after thorough testing:

1. **Create PR from `dev` → `main`**
2. **Requirements before merge**:
   - [ ] All tests passing
   - [ ] At least 2 approvals from team members
   - [ ] No merge conflicts
   - [ ] Documentation updated (if needed)
   - [ ] Release notes prepared (for major changes)
3. **Use "Squash and merge" or "Merge commit"** (team preference)
4. **Delete feature branches** after successful merge

### Best Practices

✅ **Do**:
- Keep PRs small and focused (< 400 lines when possible)
- Write descriptive commit messages
- Update `dev` regularly: `git pull origin dev`
- Test your code before creating a PR
- Respond to review comments promptly

❌ **Don't**:
- Push directly to `main` or `dev`
- Merge your own PRs without review
- Leave PRs open for more than 2-3 days
- Mix multiple unrelated changes in one PR

---

## Quick Reference Commands

```bash
# Start new feature
git checkout dev
git pull origin dev
git checkout -b feature/my-feature

# Keep your branch updated
git checkout dev
git pull origin dev
git checkout feature/my-feature
git merge dev

# Push and create PR
git push origin feature/my-feature
# Then create PR in your Git provider UI

# After PR is merged, clean up
git checkout dev
git pull origin dev
git branch -d feature/my-feature
```

---

**Questions?** Reach out to the team lead or check our repo's CONTRIBUTING.md for more details.

In [0]:
# 9. Data Analyst Summary Report
# Business Questions:
# 1. What are the top-selling categories per region?
# 2. What is the month-over-month revenue growth?
# 3. What is the average order value trend over time?

# First, let's examine the cleaned dataset
df_cleaned.printSchema()
print(f"\nTotal records: {df_cleaned.count():,}")
df_cleaned.show(5)

In [0]:
# 9. (Data Analyst) Using the cleaned dataset, produce a summary report answering 3 business questions (e.g., top-selling category per region, month-over-month growth, average order value trend) and note any data-quality caveats a stakeholder should know about.

In [0]:
from pyspark.sql.window import Window

In [0]:
df_sales = spark.read.table("samples.bakehouse.sales_transactions")
df_customers = spark.read.table("samples.bakehouse.sales_customers")

In [0]:
w = Window.partitionBy("country").orderBy(F.col("revenue").desc())

In [0]:
# Top selling products per country
import pyspark.sql.functions as F

revenue_by_country = df_sales.join(df_customers, "customerID") \
    .groupBy("country","product") \
    .agg(F.sum("totalPrice").alias("revenue")) \
    .withColumn("rank", F.dense_rank().over(w)) \
    .orderBy(F.col("country"))
display(revenue_by_country)

In [0]:
# Daily growth
w = Window.orderBy("day")
daily_growth = df_sales.groupby(F.day("dateTime").alias("day")) \
    .agg(F.sum("totalPrice").alias("revenue")) \
        .withColumn("previous_day",F.lag("revenue").over(w)) \
            .withColumn("daily_growth",F.col("revenue") - F.lag("revenue").over(w)) \
        .orderBy("day").display()

In [0]:
# Rank customer by revenue
customer_revenue = (
    df_sales
    .groupBy("customerID")
    .agg(F.sum("totalPrice").alias("revenue"))
)

w = Window.orderBy(F.col("revenue").desc())

customer_revenue_rank = (
    customer_revenue
    .withColumn("rank", F.dense_rank().over(w))
)

display(customer_revenue_rank)
